<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/05b_near_duplicate_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Do the near-duplicates change the results?

Notebook 01b flagged 51 of the 2,992 test images (1.7%) as possible near-duplicates of training or validation images.
Some are real copies, others false matches. If a real copy sits in both training and test, the CNN may have seen that test image
during training, which would make its test results look better than they should.

Instead of rebuilding the splits and retraining, this recomputes every saved result **without all 51 flagged images**, real or not,
and shows how much each number moves. If the changes are small next to the effects the thesis reports, the conclusions don't depend on the leak.
Provenance and the watermark don't learn from training data, so only the CNN (and fusion, through the CNN) can be affected.

It only rereads the saved per-image results from notebooks 05 and 01b: **no GPU, no retraining**. Runtime > Change runtime type > **CPU**. About a minute.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e .

## 2. Link the saved results on Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
for d in ["evaluation", "data_quality"]:
    !mkdir -p $D/$d && rm -rf outputs/$d && ln -s $D/$d outputs/$d
NEAR = "outputs/data_quality/near_duplicates.csv"
RUNS = ["resnet50", "efficientnet_b0", "resnet50_no_watermark", "efficientnet_b0_no_watermark"]
missing = [p for p in [NEAR] + [f"outputs/evaluation/{r}/per_image.csv" for r in RUNS] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"missing {missing}: run notebook 01b and notebook 05 (sections 4 and 7) first")
print("found the near-duplicate report and all four evaluation runs")

## 3. Recompute without the flagged images

In [ ]:
for run in RUNS:
    arch, _, rest = run.partition("_no_")
    flag = "--no-watermark" if rest else ""
    !python scripts/evaluate.py --arch $arch $flag --resummarize --exclude-near-duplicates $NEAR

## 4. How much did the results move?
**change** = result without the flagged images − result with all images. Positive accuracy change means the model did *better* without them,
so the flagged images were not inflating the score. Compare the size of the changes with the effects the thesis reports
(fusion vs CNN alone: 15–30 points; the watermark's effect on the CNN: 10–22 points).

In [ ]:
import json, pandas as pd
pd.set_option("display.width", 200)
overview = []
for run in RUNS:
    info = json.load(open(f"outputs/evaluation/{run}/summary_excl_near_duplicates.json"))
    for system, changes in info["max_abs_change"].items():
        overview.append({"run": run, "system": system, "excluded": info["excluded_images"], **{f"max |Δ {k}|": v for k, v in changes.items()}})
print("Largest change in any condition (absolute value):")
display(pd.DataFrame(overview).round(4))

for run in RUNS:
    s = pd.read_csv(f"outputs/evaluation/{run}/near_duplicate_sensitivity.csv")
    order = s.condition.unique()
    for system in ["cnn_only", "multi_signal"]:
        t = s[s.system == system].set_index("condition").loc[order, ["accuracy_all", "accuracy_without", "accuracy_change", "f1_all", "f1_without", "f1_change"]]
        print(f"\n{run} | {system}")
        display(t.round(4))

## 5. Save it
File > Save a copy in GitHub, so the results can be reviewed. The new files (`summary_excl_near_duplicates.*`, `near_duplicate_sensitivity.csv`)
are saved next to each run's results on Drive and are collected by notebook 06.